In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import os
from matplotlib import pyplot as plt
# Set working directory
os.chdir("/pscratch/sd/g/gzhao27/INR/INR_SAMPLE/")

# Verify
print("Current working directory:", os.getcwd())

from torch.utils.data import DataLoader
import numpy as np
from torch_geometric.data import Data

from torch_geometric.data import Dataset, Data
from torch.utils.data import DataLoader

from utils.data.unstructure_dataset import (
    create_ns_dataset,
    )
from utils.data.unstructure_dataset import (
    collate_graph_inr, 
    get_graph_t_idx,
    )
# sys.path.append(str(Path(__file__).parents[1]))
# sys.path.append('/pscratch/sd/g/gzhao27/INR/coral')
from utils.load_inr import create_inr_instance, load_inr_model

from torchdiffeq import odeint
from torch_geometric.data import DataLoader as GDataLoader
import numpy as np
from utils.quadtree import HierarchicalImageGrid
# from train_utility_sampling.SamplerWrapper import InrSamplerWrapper, graph_3d_cluster, graph_2d_cluster, add_cluster_label, sample_random_node_indices_per_cluster
# from mmap_ninja import RaggedMmap
# from hydra import initialize, compose

# NS_inr_save_name = 'NS_keep_for_test_file'
# NS_inr_save_dir = '/pscratch/sd/g/gzhao27/INR/SOMA/results/best_result/'
device = torch.device('cuda')


def load_inr(i, model_dir):
    inr_save_path = os.path.join(model_dir, f"{i}.pt")
    inr_results = torch.load(inr_save_path, weights_only=False)
    cfg = inr_results['cfg']

    # create & load weights
    torch.set_default_dtype(torch.float32)
    inr = create_inr_instance(cfg, input_dim=2, output_dim=1, device=device)
    inr.load_state_dict(inr_results["inr"])
    inr.to(device).eval()
    return inr 

def grad_coor(grad1, grad2):
    return torch.dot(grad1, grad2)/torch.norm(grad1)/torch.norm(grad2)

def loss_function(features_recon, features):
    loss = features_recon
    loss = ((features_recon - features)**2)
    return loss

Current working directory: /pscratch/sd/g/gzhao27/INR/INR_SAMPLE


/pscratch/sd/g/gzhao27/conda/torchgeo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# one best full run '/pscratch/sd/g/gzhao27/INR/SOMA/results/best_result/2025-08-14-12-28-58NS1024_single_null_0.0001_lr_5e-4_depth_12_end_128_t100'
# model_dir= '/pscratch/sd/g/gzhao27/INR/SOMA/results/best_result/2025-08-14-12-28-58NS1024_single_null_0.0001_lr_5e-4_depth_12_end_128_t100'
model_dir='/pscratch/sd/g/gzhao27/INR/SOMA/results/inr_sampling/2025-08-15-14-20-15NS1024_single_null_0.001_lr_5e-4_depth_6_end_128_t100'
model_path = os.path.join(model_dir, '0.pt')
save_results = torch.load(model_path, weights_only = False, map_location=device)

print('test single image inr')
cfg = save_results['cfg']
data_path = cfg.data.data_path
data_type = cfg.data.data_type  
seed = cfg.data.seed
trainset = create_ns_dataset(
            datapath = data_path, 
            data_type=data_type, 
            seed=seed,
            single_image=True  # If True, only use one image from the dataset
        )

train_loader = DataLoader(dataset=trainset, collate_fn=collate_graph_inr, batch_size=1, shuffle=False, )
graph = next(iter(train_loader))

inr = create_inr_instance(cfg, input_dim=2, output_dim=1, device=device)
t = cfg.data.single_time_frame 
indices_t = get_graph_t_idx(graph, t)

graph_ori = Data(
    cor=graph.cor[indices_t],
    feat=graph.feat[indices_t],
    time=torch.zeros(len(indices_t)),  # set time to 0 tensor
    space_emb=graph.space_emb[indices_t],
    T=torch.tensor(1),
)

files = [f for f in os.listdir(model_dir) if os.path.isfile(os.path.join(model_dir, f))]
sorted_files = sorted(files, key=lambda x: int(x.replace('.pt', '')))
print(sorted_files)

test single image inr
['0.pt', '10.pt', '20.pt', '30.pt', '40.pt', '50.pt', '60.pt', '70.pt', '80.pt', '90.pt', '100.pt', '110.pt', '120.pt', '130.pt', '140.pt', '150.pt', '160.pt', '170.pt', '180.pt', '200.pt', '210.pt', '230.pt', '250.pt', '260.pt', '270.pt', '280.pt', '290.pt', '320.pt', '330.pt', '340.pt', '360.pt', '370.pt', '400.pt', '420.pt', '430.pt', '440.pt', '450.pt', '490.pt', '510.pt', '520.pt', '530.pt', '540.pt', '550.pt', '620.pt', '630.pt', '640.pt', '650.pt', '660.pt', '670.pt', '720.pt', '730.pt', '740.pt', '750.pt', '760.pt', '860.pt', '870.pt', '880.pt', '890.pt', '900.pt', '950.pt', '960.pt', '970.pt', '980.pt', '1090.pt', '1100.pt', '1120.pt', '1140.pt', '1150.pt', '1160.pt', '1170.pt', '1180.pt', '1190.pt', '1270.pt', '1280.pt', '1290.pt', '1300.pt', '1370.pt', '1380.pt', '1460.pt', '1470.pt', '1480.pt', '1490.pt', '1580.pt', '1590.pt', '1600.pt', '1610.pt', '1620.pt', '1710.pt', '1720.pt', '1840.pt', '1850.pt', '1940.pt', '1950.pt', '2140.pt', '2150.pt', '3370.

In [4]:
# per pixel loss calculation
t = 100

torch.cuda.empty_cache()
graph = graph_ori.to(device)
H = graph.cor.max().item()+1

features = graph.feat.view(H, H, 1)
coords = graph.space_emb.detach().view(H, H, 2)

rate = 1

center = (341, 341)

features = features[::rate, ::rate]
coords = coords[::rate, ::rate]


inr = load_inr(t, model_dir)
inr.to(device)
reconfeature = inr(coords)

params = list(inr.parameters())
per_pix_losses = loss_function(reconfeature, features)

### calculate batch sample grad use jacrev and iterations

In [5]:
from torch.func import jacrev, vmap, functional_call

def per_sample_loss(inr, params, coords, targets):
    """Compute scalar loss per sample (no reduction)."""
    recon = functional_call(inr, params, (coords,))
    losses = loss_function(recon, targets)  # [B]
    return losses

def per_sample_grad_norms(inr, params, coords, targets):
    """Return [B] tensor of squared gradient norms."""
    
    # function: params -> loss[i]
    def loss_i_fn(p, x, y):
        return per_sample_loss(inr, p, x.unsqueeze(0), y.unsqueeze(0)).squeeze(0)

    # Compute per-sample gradients via vmap
    grads = vmap(jacrev(loss_i_fn), in_dims=(None, 0, 0))(params, coords, targets)
    # grads is a PyTree: list of tensors [B, ...] same shapes as params
    
    # Flatten per-sample gradients and compute norm
    batch_size = next(iter(grads.values())).shape[0]
    norms = torch.zeros(batch_size, device=coords.device)

    for g in grads.values():
        g = g.reshape(batch_size, -1)   # flatten per-sample grads
        norms += (g ** 2).sum(dim=1)    # add squared norm per sample

    return norms.sqrt()

from collections import OrderedDict

# Turn model parameters into a dict
params = OrderedDict((name, p) for name, p in inr.named_parameters())

coords_sub = coords[:100, :40]
targets_sub = features[:100, :40]
from time import time
start = time()
pgrad = per_sample_grad_norms(inr, params, coords_sub.reshape(-1, 2), targets_sub.reshape(-1))
print(time()-start)

0.2189958095550537


In [6]:
def per_sample_grad_norms_true(inr, params, coords, targets):
    """
    Compute per-sample squared gradient norms: [B]
    """
    params = list(inr.parameters())
    reconfeature = inr(coords)
    per_pix_losses = loss_function(reconfeature, targets)

    H, W = targets.shape[0], targets.shape[1]
    all_pix_grads = []
    for i in range(H):
        for j in range(W):
            loss = per_pix_losses[i, j]
            point_grad = torch.autograd.grad(
                loss, params, retain_graph=True, allow_unused=True
            )
            grads_all = [g.view(-1) for g in point_grad if g is not None]
            grad_vec = torch.cat(grads_all)
            
            all_pix_grads.append(grad_vec.norm().cpu().item())

    return all_pix_grads

from time import time
start = time()
pgrad2 = per_sample_grad_norms_true(inr, params, coords_sub, targets_sub)
print(time()-start)

4.309762239456177


### second order grad frobenius norm estimate 

In [9]:
import torch
from train_utility_sampling.taylor_estimation import (
estimate_frobenius_norm_corrected, 
grad_estimation, 
grad_estimation_fully_batched, 
estimate_frobenius_norm_swaped,
loss_function, 
frobenius_norm_via_jacrev,
)
from time import time
r, c = 1000, 16
graph = graph.to(device)
features = graph.feat.view(1024, 1024, 1)
coords = graph.space_emb.detach().view(1024, 1024, 2).requires_grad_(True)
s_step = 2/1024. # r+1 and r-1 distance
y_x = torch.tensor([(features[r+1, c] - features[r-1, c])/s_step, (features[r, c+1] - features[r, c-1])/s_step]).to(device)/2


i = 100
inr = load_inr(i, model_dir)
inr.to(device)
params = list(inr.parameters())

input_x = coords[r, c].clone().detach().to(device).requires_grad_() 
features_recon_point = inr(input_x)
loss = loss_function(features_recon_point, features[r, c])
fro_norm = estimate_frobenius_norm_corrected(loss, features_recon_point, params, input_x, y_x, 500)


start = time()
print('estimate_frobenius_norm_corrected')
for i in range(5):
    fro_norm = estimate_frobenius_norm_corrected(loss, features_recon_point, params, input_x, y_x, 500)
    print(fro_norm)
print(time()-start)

estimate_frobenius_norm_corrected
tensor(63891.5742, device='cuda:0', grad_fn=<SqrtBackward0>)
tensor(65964.4062, device='cuda:0', grad_fn=<SqrtBackward0>)
tensor(65942.7266, device='cuda:0', grad_fn=<SqrtBackward0>)
tensor(65276.7383, device='cuda:0', grad_fn=<SqrtBackward0>)
tensor(68782.5391, device='cuda:0', grad_fn=<SqrtBackward0>)
5.155677556991577


In [10]:
features_recon = inr(coords)
loss_per_pixel = loss_function(features_recon, features)

loss_x = torch.stack([
        (loss_per_pixel[r+1, c] - loss_per_pixel[r-1, c]) / s_step,
        (loss_per_pixel[r, c+1] - loss_per_pixel[r, c-1]) / s_step
    ], dim=1).to(device) / 2

start = time()
params = list(inr.parameters())
print('estimate_frobenius_norm_swaped')
for i in range(5):
    fro_norm2 = estimate_frobenius_norm_swaped(loss_x, params, device)
    print(fro_norm2)
print(time()-start)


estimate_frobenius_norm_swaped
tensor([66770.8594], device='cuda:0')
tensor([66770.8594], device='cuda:0')
tensor([66770.8594], device='cuda:0')
tensor([66770.8594], device='cuda:0')
tensor([66770.8594], device='cuda:0')
0.49819183349609375


In [34]:
H = graph.cor.max().item()+1
coords_range = coords.max() - coords.min()
coords_step = coords_range / (H-1)
neighbor_coords = []
neighbor_targets = []
neighbors = [
    (r + 1, c),
    (r - 1, c),
    (r, c + 1),
    (r, c - 1),
]
for rr, cc in neighbors:
    neighbor_coords.append(coords[rr, cc])
    neighbor_targets.append(features[rr, cc])

neighbor_coords = torch.stack(neighbor_coords, dim=0)    # [4B, 2]
neighbor_targets = torch.stack(neighbor_targets, dim=0) 
start = time()

print('frobenius_norm_via_jacrev')
for i in range(5):
    fro_norm3 = frobenius_norm_via_jacrev(inr, params, neighbor_coords, neighbor_targets, coords_step)
    print(fro_norm3)
print(time()-start)

frobenius_norm_via_jacrev
tensor([66705.2500], device='cuda:0', grad_fn=<SqrtBackward0>)
tensor([66705.2500], device='cuda:0', grad_fn=<SqrtBackward0>)
tensor([66705.2500], device='cuda:0', grad_fn=<SqrtBackward0>)
tensor([66705.2500], device='cuda:0', grad_fn=<SqrtBackward0>)
tensor([66705.2500], device='cuda:0', grad_fn=<SqrtBackward0>)
0.02904653549194336


### within cell gradient variance estimation

In [13]:
from train_utility_sampling.taylor_estimation import (
    grad_variance_ground_truth,
    cell_grad_variance_estimate_with_norm_corrected,
    cell_grad_variance_estimate_with_jacrev,
    loss_variance_ground_truth,
)
r = 1000
c = 400
cell_range = 8
if cell_range % 2 ==1:
    cell_cor_range = [r - cell_range//2, r + cell_range//2 , c - cell_range//2, c + cell_range//2 ] # odd number setting
else:
    cell_cor_range = [r - cell_range//2, r + cell_range//2 -1 , c - cell_range//2, c + cell_range//2 -1 ]
cell_cor_range = torch.tensor([cell_cor_range]).to(device)

In [19]:
grad_variance_ground_truth(cell_cor_range[0], loss_per_pixel, params, graph)

ValueError: not enough values to unpack (expected 4, got 1)

In [23]:
loss_variance_ground_truth(cell_cor_range, graph, inr, device)

tensor([0.0039], device='cuda:0')

In [224]:
width = (cell_range-1)* coords_step
width**2 /12 * (cell_range +1 ) / (cell_range-1)

tensor(2.0066e-05, device='cuda:0', grad_fn=<DivBackward0>)

In [211]:
embbb =graph.space_emb.reshape(1024, 1024, 2)

rupper = cell_cor_range[0,1].item()
rlower = cell_cor_range[0,0].item()
cupper = cell_cor_range[0,3].item()
clower = cell_cor_range[0,2].item()

all_emb = embbb[rlower:rupper+1, clower:cupper+1]
all_emb.var(dim=(0,1))
        # calculate embbb[r,c] variance
        

tensor([2.5800e-05, 2.5800e-05])

In [1]:
cell_grad_variance_estimate_with_jacrev(cell_cor_range, graph, inr, device)

NameError: name 'cell_grad_variance_estimate_with_jacrev' is not defined

## test grid with cell_grad_variance_estimate_with_jacrev